In [1]:
import os

import numpy as np
import pennylane as qml
import tensorflow as tf
from scipy.io import arff
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from ucimlrepo import fetch_ucirepo

from scr.qsvdd_core.data_loader import QuantumDataLoader
from scripts.test_model import mean_auc
from scripts.train_model import train_five_times

In [2]:
np.random.seed(42)

In [3]:
n_train = 0 ; latent_dim = 3
# num_params_conv = 375
cost_func = 'svdd'

# Breast Cancer anomaly Detection

In [38]:
print("="*60)
print("Loading and processing the Breast Cancer dataset...")
print("="*60)

# 1. Fetch the dataset
breast_cancer = fetch_ucirepo(id=17)
X_bc_raw = breast_cancer.data.features
y_bc_raw = breast_cancer.data.targets

Loading and processing the Breast Cancer dataset...


In [ ]:
print(breast_cancer.data.features.head())

In [ ]:
X_bc_raw.info()

In [ ]:
y_bc_raw.info()

In [27]:
y_bc = y_bc_raw.iloc[:, 0].apply(lambda x: 1 if x == 'M' else 0).values

loader = QuantumDataLoader()

import pandas as pd
df_bc = pd.DataFrame(X_bc_raw)
df_bc['Class'] = y_bc

X_data = loader.prepare_bc_data(X_bc_raw)
y = y_bc

# 4. Identify indices for each class
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# 5. Training Set (One-Class: 200 normal samples)
X_train_normal_indices = np.random.choice(normal_indices, 250, replace=False)
X_train = X_data[X_train_normal_indices]
Y_train = y[X_train_normal_indices]

# 6. Test Set (Balanced: 50 normal + 50 anomalies)
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))

X_test_normal_indices = np.random.choice(remaining_normal_indices, 50, replace=False)
X_test_abnormal_indices = np.random.choice(abnormal_indices, 50, replace=False)

X_test_normal = X_data[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

X_test_abnormal = X_data[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# Final test set (Mix: 100 samples)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

print(f"X_train shape (Normal): {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape (Mix): {X_test.shape}")
print(f"Y_test shape: {Y_test.shape}")

X_train shape (Normal): (250, 32)
Y_train shape: (250,)
X_test shape (Mix): (100, 32)
Y_test shape: (100,)


In [28]:
loader = QuantumDataLoader()

X_quantum = loader.prepare_bc_data(X_bc_raw)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (569, 32)
Labels: (569,)


# ALOI Dataset

In [14]:
data, meta = arff.loadarff('../data/ALOI_withoutdupl_norm.arff')
df = pd.DataFrame(data)

# Pegamos da primeira coluna até a 'att27' (índice 0 até 26)
# O fatiamento :27 pega os índices de 0 a 26.
X = df.iloc[:, :27].values

# 3. Tratar o Label (y)
# A coluna de label chama-se 'outlier'
y_raw = df['outlier']

# Converter bytes para string e depois para binário (yes=1, no=0)
y = y_raw.apply(lambda x: x.decode('utf-8').lower() if isinstance(x, bytes) else str(x).lower())
y = np.where(y == 'yes', 1, 0)

# 4. Resultados Finais
print(f"{'='*30}")
print(f"DATASET ALOI CARREGADO")
print(f"{'='*30}")
print(f"Instâncias: {X.shape[0]}")
print(f"Atributos (Features): {X.shape[1]} (att1 até att27)")
print(f"Anomalias (Outliers): {np.sum(y)}")
print(f"Proporção de Outliers: {np.mean(y)*100:.2f}%")
print(f"{'='*30}")

DATASET ALOI CARREGADO
Instâncias: 49534
Atributos (Features): 27 (att1 até att27)
Anomalias (Outliers): 1508
Proporção de Outliers: 3.04%


In [17]:
def prepare_aloi_classic(df):
    # No ALOI, as colunas são 'att1'...'att27', o label é 'outlier' e tem o 'id'
    # Selecionamos as 27 colunas de atributos
    X = df.iloc[:, :27].values

    # Tratamos o label 'outlier' para 0 e 1
    y_raw = df['outlier'].apply(lambda x: x.decode('utf-8').lower() if isinstance(x, bytes) else str(x).lower())
    y = np.where(y_raw == 'yes', 1, 0)

    # Aplicar o Scaler (opcional para ALOI, mas mantém consistência com seu projeto)
    scaler = StandardScaler()
    X_classic = scaler.fit_transform(X)

    X_padded = np.pad(X_classic, ((0, 0), (0, 5)), mode='constant', constant_values=0)
    print(f"Shape para o QSVDD: {X_padded.shape}") # (49534, 32)
    # troca x_classic por x_padded no return se for quantum_algorithm
    return X_padded, y


# Executando a preparação
X_quantum, y = prepare_aloi_classic(df)

print(f"Features para o modelo: {X_data.shape}")
print(f"Labels: {y.shape}")
print(f"Total de Anomalias: {np.sum(y)}")

Shape para o QSVDD: (49534, 32)
Features para o modelo: (49534, 32)
Labels: (49534,)
Total de Anomalias: 1508


# Credit Card

In [39]:
file_path = os.path.join("..", "data", "creditcard.csv")
df = pd.read_csv(file_path)

print("Dataset loaded successfully!\n")
print("First 5 records:\n", df.head())

Dataset loaded successfully!

First 5 records:
    Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.14126

In [40]:
tmp = df[['Amount','Class']].copy()
class_0 = tmp.loc[tmp['Class'] == 0]['Amount']
class_1 = tmp.loc[tmp['Class'] == 1]['Amount']

In [41]:
loader = QuantumDataLoader()

X_quantum, y = loader.prepare_fraud_data(df)

print(f"Features for the circuit: {X_quantum.shape}")
print(f"Labels: {y.shape}")

Features for the circuit: (284807, 32)
Labels: (284807,)


# MNIST

In [4]:
# Declare the normal class and the dimension of the latent space
# For QSVDD: num_params_conv <- 15, cost_func <- 'svdd', steps = 500
# For QAE: num_params_conv <- 8, cost_func <- 'qae', steps = 2000

# dataset <- 'mnist', 'fmnist', 'cifar'
# ntrain <- number of class want to train
# latent_dim <- 3, 6, 9, 12, 15

dataset = 'mnist'
ntrain = 0 ; latent_dim = 3
num_params_conv = 15
cost_func = 'svdd'

In [5]:
# Import data
def data(ntrain, latent_dim, dataset):
    # Load dataset
    if dataset == 'mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    elif dataset == 'fmnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

    # Normalize the data
    x_train, x_test = x_train[..., np.newaxis] / 255.0, x_test[..., np.newaxis] / 255.0

    # Filter training data (only normal class)
    train_idx = np.where(y_train == ntrain)[0]
    X_train = x_train[train_idx]

    # Sample 600 training instances
    train_indices = np.random.choice(len(X_train), 600, replace=False)
    X_train = X_train[train_indices]
    Y_train = np.zeros(600)  # Label 0 for normal class

    # Filter test data indices (normal and anomalous classes)
    normal_idx = np.where(y_test == ntrain)[0]
    anomaly_idx = np.where(y_test != ntrain)[0]

    # Sample 50 normal and 50 anomaly instances
    normal_samples = np.random.choice(normal_idx, 50, replace=False)
    anomaly_samples = np.random.choice(anomaly_idx, 50, replace=False)

    # Concatenate test samples and create labels (0 = normal, 1 = anomaly)
    test_indices = np.concatenate([normal_samples, anomaly_samples])
    X_test = x_test[test_indices]
    Y_test = np.concatenate([np.zeros(50), np.ones(50)])

    # Shuffle test data to avoid order bias
    shuffle_idx = np.random.permutation(100)
    X_test = X_test[shuffle_idx]
    Y_test = Y_test[shuffle_idx]

    # Resize data
    X_train = tf.image.resize(X_train[:], (256, 1)).numpy()
    X_test = tf.image.resize(X_test[:], (256, 1)).numpy()
    X_train, X_test = tf.squeeze(X_train).numpy(), tf.squeeze(X_test).numpy()

    x_train = tf.image.resize(x_train[:], (256, 1)).numpy()
    x_test = tf.image.resize(x_test[:], (256, 1)).numpy()
    x_train, x_test = tf.squeeze(x_train).numpy(), tf.squeeze(x_test).numpy()

    # Quantum setup (center)
    center = qml.numpy.zeros(latent_dim, requires_grad=True)
    center_train = np.tile(center, (len(X_train), 1))

    # Print shapes to verify
    print('x_train:', x_train.shape)
    print('x_test:', x_test.shape)
    print('X_train:', X_train.shape)
    print('X_test:', X_test.shape)
    print('Y_train:', Y_train.shape)
    print('Y_test:', Y_test.shape)

    return x_train, y_train, x_test, y_test, X_train, X_test, Y_train, Y_test, center_train

In [14]:
x_train, y_train, x_test, y_test, X_train, X_test, Y_train, Y_test, center_train = data(ntrain, latent_dim, dataset)

x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)


# Statlog (Shuttle) Dataset

In [4]:
# Fetch dataset
statlog_shuttle = fetch_ucirepo(id=148)

# Load data (as pandas dataframes)
x_shuttle = statlog_shuttle.data.features
y_shuttle = statlog_shuttle.data.targets

# Binarize labels: Class 1 becomes 0 (Normal), others become 1 (Anomaly)
ysb = y_shuttle.iloc[:, 0].apply(lambda val: 0 if val == 1 else 1)

# Scale features
scaller = MinMaxScaler()
x_scaled = scaller.fit_transform(x_shuttle)

# Pad with zeros to match qubit requirements
x_padded = np.pad(
    x_scaled,
    ((0, 0), (0, 1)),
    mode="constant",
    constant_values=0
)

# Normalize vectors
norms = np.linalg.norm(x_padded, axis=1, keepdims=True)

# Final quantum data and fixed label assignment
X_quantum = x_padded / norms
y = ysb.to_numpy() # Converted to numpy array to prevent indexing issues later

# Fix undefined variable 'x_data'
n_samples, n_features = X_quantum.shape

print(n_samples, n_features)

58000 8


# DSVDD

In [6]:
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score

# Dictionary to store results for each run
results = {
    "Isolation Forest": [],
    "One-Class SVM": [],
    "LOF": []
}

n_runs = 5

for i in range(n_runs):

    _, _, _, _, X_train_run, X_test_run, _, Y_test_run, _ = data(0, 8, 'mnist')

    # 2. Instantiate Models
    models = {
        "Isolation Forest": IsolationForest(n_estimators=100, contamination=0.01, random_state=42),
        "One-Class SVM": OneClassSVM(kernel='rbf', nu=0.1, gamma='scale'),
        "LOF": LocalOutlierFactor(n_neighbors=20, novelty=True)
    }

    # 3. Train and Evaluate
    for name, model in models.items():
        model.fit(X_train_run)

        # For AUC, we need decision scores rather than binary classes
        if name == "LOF":
            # Invert scores as lower values indicate anomalies in LOF
            y_scores = -model.decision_function(X_test_run)
        else:
            # Invert decision_function (which returns negative for anomalies)
            # so that higher scores = higher probability of fraud
            y_scores = -model.decision_function(X_test_run)

        auc = roc_auc_score(Y_test_run, y_scores)
        results[name].append(auc)

# 4. Display Means and Standard Deviation in a formatted table
print(f"\n{'='*55}")
print(f"{'Classical Algorithm':<25} | {'Mean AUC':<12} | {'Std AUC':<10}")
print(f"{'-'*25}-|-{'-'*12}-|-{'-'*10}")

for name, scores in results.items():
    mean_auc = np.mean(scores)
    std_auc = np.std(scores)
    print(f"{name:<25} | {mean_auc:<12.4f} | {std_auc:<10.4f}")

print(f"{'='*55}\n")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)

Classical Algorithm       | Mean AUC     | Std AUC   
--------------------------|--------------|-----------
Isolation Forest          | 0.9768       | 0.0100    
One-Class SVM             | 0.9281       | 0.0358    
LOF                       | 0.9769       | 0.0140    



In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Configuração de Hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Definição da Arquitetura do Modelo
class SVDD(nn.Module):
    def __init__(self, latent_dim, num_filters=3):
        super(SVDD, self).__init__()
        self.num_filters = num_filters
        self.latent_dim = latent_dim
        # Conv1d para dados que foram redimensionados para (256, 1)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=self.num_filters, kernel_size=2, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(num_features=self.num_filters)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(in_channels=self.num_filters, out_channels=self.num_filters, kernel_size=2, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(num_features=self.num_filters)
        self.conv3 = nn.Conv1d(in_channels=self.num_filters, out_channels=2, kernel_size=2, stride=1, padding=1)
        self.bn3 = nn.BatchNorm1d(num_features=2)

        # Ajuste o input da linear conforme o output da última pool
        self.fc1 = nn.Linear(64, self.latent_dim) # O valor 8 pode variar dependendo do resize exato

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

# 3. Função para inicializar o centro C (ponto central da hiperesfera)
def initialize_center_c(dataloader, model):
    model.eval()
    center = torch.zeros(model.latent_dim, device=device)
    n_samples = 0
    with torch.no_grad():
        for inputs, _ in dataloader:
            inputs = inputs.unsqueeze(1).to(device)
            outputs = model(inputs)
            center += torch.sum(outputs, dim=0)
            n_samples += outputs.shape[0]

    center /= n_samples

    # Se o centro estiver muito próximo de 0, forçamos um pequeno deslocamento
    # para evitar soluções triviais (colapso da rede)
    eps = 0.1
    center[(center > -eps) & (center < eps)] += eps

    model.train()
    return center

# ---------------------------------------------------------
# 4. Loop Principal de Experimentos (5 rodadas)
# ---------------------------------------------------------
dsvdd_results = []
n_runs = 5
latent_dim = 3

for run in range(n_runs):
    print(f"\n--- Iniciando Rodada Deep SVDD {run+1}/{n_runs} ---")

    # Gerar dados novos
    _, _, _, _, X_train_run, X_test_run, Y_train_run, Y_test_run, _ = data(0, latent_dim, 'mnist')

    # Preparar tensores
    x_train_tensor = torch.tensor(X_train_run, dtype=torch.float32)
    y_train_tensor = torch.tensor(Y_train_run, dtype=torch.float32)
    x_test_tensor = torch.tensor(X_test_run, dtype=torch.float32)
    y_test_tensor = torch.tensor(Y_test_run, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(x_train_tensor, y_train_tensor), batch_size=16, shuffle=True)
    test_loader = DataLoader(TensorDataset(x_test_tensor, y_test_tensor), batch_size=1)

    # Reinicializar modelo e otimizador
    svdd = SVDD(latent_dim=latent_dim).to(device)
    optimizer = torch.optim.Adam(svdd.parameters(), lr=0.001, weight_decay=1e-6)

    # Passo crucial: Inicializar o Centro C com os pesos iniciais da rede
    c = initialize_center_c(train_loader, svdd)

    # Treinamento
    svdd.train()
    for step in range(500):
        try:
            inputs, _ = next(iter_loader)
        except (NameError, StopIteration):
            iter_loader = iter(train_loader)
            inputs, _ = next(iter_loader)

        inputs = inputs.unsqueeze(1).to(device)
        optimizer.zero_grad()

        outputs = svdd(inputs)
        # Loss: Distância Euclidiana Quadrática até o centro
        loss = torch.mean(torch.sum((outputs - c) ** 2, dim=1))

        loss.backward()
        optimizer.step()

        if step % 100 == 0:
            print(f'Step: {step} | Loss: {loss.item():.6f}')

    # Avaliação
    svdd.eval()
    distances = []
    labels = []
    with torch.no_grad():
        for inputs, target in test_loader:
            inputs = inputs.unsqueeze(1).to(device)
            outputs = svdd(inputs)
            dist = torch.sum((outputs - c) ** 2, dim=1)
            distances.append(dist.cpu().item())
            labels.append(target.item())

    auc_score = roc_auc_score(labels, distances)
    dsvdd_results.append(auc_score)
    print(f"Rodada {run+1} Finalizada. AUC: {auc_score:.4f}")

# 5. Resultado Final
print("\n" + "="*55)
print(f"{'Algorithm':<25} | {'Mean AUC':<12} | {'Std AUC':<10}")
print(f"{'-'*25}-|-{'-'*12}-|-{'-'*10}")
print(f"{'Deep SVDD':<25} | {np.mean(dsvdd_results):<12.4f} | {np.std(dsvdd_results):<10.4f}")
print("="*55)


--- Iniciando Rodada Deep SVDD 1/5 ---
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
Step: 0 | Loss: 0.374555
Step: 100 | Loss: 0.010231
Step: 200 | Loss: 0.007871
Step: 300 | Loss: 0.003135
Step: 400 | Loss: 0.001204
Rodada 1 Finalizada. AUC: 0.7780

--- Iniciando Rodada Deep SVDD 2/5 ---
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
Step: 0 | Loss: 0.327298
Step: 100 | Loss: 0.024987
Step: 200 | Loss: 0.011799
Step: 300 | Loss: 0.008419
Step: 400 | Loss: 0.002933
Rodada 2 Finalizada. AUC: 0.9156

--- Iniciando Rodada Deep SVDD 3/5 ---
x_train: (60000, 256)
x_test: (10000, 256)
X_train: (600, 256)
X_test: (100, 256)
Y_train: (600,)
Y_test: (100,)
Step: 0 | Loss: 0.054062
Step: 100 | Loss: 0.005401
Step: 200 | Loss: 0.002270
Step: 300 | Loss: 0.001729
Step: 400 | Loss: 0.001529
Rodada 3 Finalizada. AUC: 0.9228

--- Iniciando Rodada Deep SVDD 4/5 ---
x_t

# Preparing Quantum Data

In [5]:
"""
One-class Training:
Separation into normal and fraudulent examples
QSVDD will learn what is normal.
"""
normal_indices = np.where(y == 0)[0]
abnormal_indices = np.where(y == 1)[0]
np.random.seed(42)

# Train
# collect 1000 normal examples for training
X_train_normal_indices = np.random.choice(normal_indices, 1000, replace=False)
X_train_normal = X_quantum[X_train_normal_indices]
y_train_normal = y[X_train_normal_indices]

# X_train contains only legitimate transactions
X_train = X_train_normal
Y_train = y_train_normal

# Test (balanced)
# The code removes 100 normal examples that were not used in training
remaining_normal_indices = list(set(normal_indices) - set(X_train_normal_indices))
X_test_normal_indices = np.random.choice(remaining_normal_indices, 100, replace=False)
X_test_normal = X_quantum[X_test_normal_indices]
y_test_normal = y[X_test_normal_indices]

# The code removes 100 fraud examples.
X_test_abnormal_indices = np.random.choice(abnormal_indices, 100, replace=False)
X_test_abnormal = X_quantum[X_test_abnormal_indices]
y_test_abnormal = y[X_test_abnormal_indices]

# creates a test set with 200 examples (50% normal, 50% fraud)
X_test = np.concatenate((X_test_normal, X_test_abnormal), axis=0)
Y_test = np.concatenate((y_test_normal, y_test_abnormal), axis=0)

center = np.zeros(latent_dim)
center_train = np.tile(center, (len(X_train), 1))

print(f'X_train shape: {X_train.shape}')
print(f'Y_train shape: {Y_train.shape}')
print(f'X_test_normal shape: {X_test_normal.shape}')
print(f'X_test_abnormal shape: {X_test_abnormal.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Y_test shape: {Y_test.shape}')
print(f'center_train shape: {center_train.shape}')

X_train shape: (1000, 8)
Y_train shape: (1000,)
X_test_normal shape: (100, 8)
X_test_abnormal shape: (100, 8)
X_test shape: (200, 8)
Y_test shape: (200,)
center_train shape: (1000, 3)


In [6]:
train_Xdata = X_train
train_Ydata = center_train

### QCNN (Quantum Convolutional Neural Network) Ansatz

#### Hyperparameters

In [11]:
qcnn_batch_size = 4
qcnn_steps = 2000
qcnn_learning_rate = 0.01

In [12]:
(qcnn_loss_history_matrix,
 qcnn_est_params_matrix,
 qcnn_param_history_matrix,
 qcnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qcnn_batch_size,
                                        learning_rate=qcnn_learning_rate,
                                        steps=qcnn_steps,
                                        ansatz='qcnn'
                                        )
loss_history_f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qcnn_loss_history_matrix)
np.savetxt(est_params_f_name, qcnn_est_params_matrix)
np.savetxt(time_f_name, qcnn_time_record)

print("--- All training batches completed ---")

--- Starting training round 1 with seed 1617597454 ---


/home/jvfg/Documents/ORG/Repos/QSVDD2/.venv/lib/python3.12/site-packages/autograd/numpy/numpy_vjps.py:943: ComplexWarning: Casting complex values to real discards the imaginary part
  onp.add.at(A, idx, x)


--- Starting training round 2 with seed 200160611 ---
--- Starting training round 3 with seed 120710055 ---
--- Starting training round 4 with seed 3182564434 ---
--- Starting training round 5 with seed 356766577 ---
--- All training batches completed ---


#### QCNN Training Evaluation

In [13]:
f_name = f"../results/training/QCNN/MISC_QCNN_B{qcnn_batch_size:02d}S{qcnn_steps}LR{qcnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qcnn")

print(50*"--")
print(f'for: B{qcnn_batch_size}S{qcnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.70s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.69s
Test completed in 1.40s | AUC: 0.8518
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.69s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.68s
Test completed in 1.38s | AUC: 0.8163
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.70s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.69s
Test completed in 1.39s | AUC: 0.6504
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.69s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.73s
Test completed in 1.43s | AUC: 0.7892
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.71s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.71s
Test completed in 1.42s | AUC: 0.7985
----------------------------------------------------------------------------------------------------
for: B4S2

### QAE (Quantum AutoEncoder) Ansatz

In [19]:
qae_batch_size = 16
qae_steps = 500
qae_learning_rate = 0.001

#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [20]:
(qae_loss_history_matrix,
 qae_est_params_matrix,
 qae_param_history_matrix,
 qae_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=qae_batch_size,
                                        learning_rate=qae_learning_rate,
                                        steps=qae_steps,
                                        ansatz='qae'
                                        )
loss_history_f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, qae_loss_history_matrix)
np.savetxt(est_params_f_name, qae_est_params_matrix)
np.savetxt(time_f_name, qae_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 1775299509 ---
--- Starting training round 2 with seed 910200048 ---
--- Starting training round 3 with seed 2174764617 ---
--- Starting training round 4 with seed 2938630505 ---
--- Starting training round 5 with seed 2656783416 ---
--- All training batches completed ---


#### QAE Training Evaluation

In [21]:
f_name = f"../results/training/QAE/MISC_QAE_B{qae_batch_size:02d}S{qae_steps}LR{qae_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="qae")

print(50*"--")
print(f'for: B{qae_batch_size}S{qae_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.23s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.46s | AUC: 0.8896
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.45s | AUC: 0.8281
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.23s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.45s | AUC: 0.7825
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.23s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.46s | AUC: 0.9023
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.44s | AUC: 0.8639
----------------------------------------------------------------------------------------------------
for: B16S

### LCQHNN (Lean classical-quantum hybrid neural network) Ansatz

The latent space for this ansatz has dimension 5 (vs. 3 for QCNN/QAE), requiring a re-initialized center vector.

In [8]:
center = np.zeros(3)
center_train = np.tile(center, (len(X_train), 1))
lcqhnn_batch_size = 8
lcqhnn_steps = 2000
lcqhnn_learning_rate = 0.01
print(f'center_train shape: {center_train.shape}')
train_Xdata = X_train
train_Ydata = center_train

center_train shape: (1000, 3)


#### Five-Run Training & Result Persistence

Run `train_five_times` to train the **QAE** ansatz across 5 independent runs with the configured batch size and step count.

In [9]:
(lcqhnn_loss_history_matrix,
 lcqhnn_est_params_matrix,
 lcqhnn_param_history_matrix,
 lcqhnn_time_record) = train_five_times(X_train=train_Xdata,
                                        Y_train=train_Ydata,
                                        batch_size=lcqhnn_batch_size,
                                        learning_rate=lcqhnn_learning_rate,
                                        steps=lcqhnn_steps,
                                        ansatz='lcqhnn'
                                        )
loss_history_f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_LOSS_HISTORY_MEAN.npy"
est_params_f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
time_f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_TIME_MEAN.npy"
np.savetxt(loss_history_f_name, lcqhnn_loss_history_matrix)
np.savetxt(est_params_f_name, lcqhnn_est_params_matrix)
np.savetxt(time_f_name, lcqhnn_time_record)
print("--- All training batches completed ---")

--- Starting training round 1 with seed 3484775655 ---
--- Starting training round 2 with seed 3786391805 ---
--- Starting training round 3 with seed 3915279021 ---
--- Starting training round 4 with seed 3280553813 ---
--- Starting training round 5 with seed 1159680386 ---
--- All training batches completed ---


#### LCQHNN Training Evaluation

In [10]:
f_name = f"../results/training/LCQHNN/MISC_LCQHNN_B{lcqhnn_batch_size:02d}S{lcqhnn_steps}LR{lcqhnn_learning_rate:.0e}_EST_PARAMS_MEAN.npy"
params_list = np.loadtxt(f_name)
mean, std = mean_auc(params_list, n_train, X_test, Y_test, center_train, noisy=False, ansatz="lcqhnn")

print(50*"--")
print(f'for: B{lcqhnn_batch_size}S{lcqhnn_steps} | AUC_mean: {mean} | std: {std}')
print(50*"--")

Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.24s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.46s | AUC: 0.9242
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.22s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.44s | AUC: 0.9670
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.21s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.23s
Test completed in 0.44s | AUC: 0.8338
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.20s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.22s
Test completed in 0.43s | AUC: 0.9157
Processing class 0 (label 0) | Samples: 100
Finished class 0 in 0.21s
Processing class 1 (label 1) | Samples: 100
Finished class 1 in 0.21s
Test completed in 0.42s | AUC: 0.8078
----------------------------------------------------------------------------------------------------
for: B8S2

## Noisy (NISQ-Era) Training

In the noisy setting, the quantum circuits are simulated with a **hardware noise model** that mimics real NISQ (Noisy Intermediate-Scale Quantum) device behavior, including gate errors and decoherence. This evaluates model robustness under realistic quantum hardware conditions.

The same three ansatzes are retrained from scratch under this noisy simulation.